# 06. 동적 페이지 수집
로그인, 명시적 대기, 페이지 이동과 여러 페이지 반복 수집을 하나의 흐름으로 정리합니다.

#### 동적 스크래핑 실습 사이트

- https://quotes.toscrape.com/
- 유명인사들의 명언 등을 보여주는 사이트
- 스크래핑을 위해 개발된 샘플 사이트

In [ ]:
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://quotes.toscrape.com/')

footer = wait.until(
    EC.presence_of_element_located((By.CLASS_NAME, 'footer'))
)

##### 자동 로그인 연습

- wait.until() : 다음 페이지 화면이 전환됬을때 모두 표시되기전에 스크래핑 시작하면 오류가 발생. 화면이 무조건 다 로드이후 작업 시작하라는 의미
- time.sleep(10) 보다 훨씬 효율적

In [ ]:
# 로그인 링크 찾아서 클릭
login_link = wait.until(
    EC.element_to_be_clickable((By.LINK_TEXT, 'Login'))
)

login_link.click()

In [ ]:
# 아이디 패스워드 입력
username_input = wait.until(
    EC.presence_of_element_located((By.ID, 'username'))
)

password_input = driver.find_element(By.ID, 'password')

username_input.send_keys('admin')
password_input.send_keys('admin')

In [ ]:
# 로그인버튼 클릭
login_btn = driver.find_element(By.CSS_SELECTOR, 'input[type="submit"]')

login_btn.click()

In [ ]:
# 로그인이 완료된 것 확인
wait.until(
    EC.presence_of_element_located((By.LINK_TEXT, 'Logout'))
)

print('로그인 성공 : 작업 시작')

##### 인용문 찾기

In [ ]:
quotes = driver.find_elements(By.CLASS_NAME, 'quote')

len(quotes)

In [ ]:
# 첫번째 인용문 확인
quotes = driver.find_elements(By.CLASS_NAME, 'quote')

first = quotes[3]

text = first.find_element(By.CSS_SELECTOR, '.text').text
author = first.find_element(By.CSS_SELECTOR, '.author').text
link = first.find_elements(By.CSS_SELECTOR, 'span > a')[1].get_attribute('href')
tags = first.find_elements(By.CSS_SELECTOR, '.tags > a')
tag_text = ', '.join(tag.text for tag in tags)

print(text)
print(author)
print(link)
print(tag_text)

##### 한 페이지 수집하기

In [ ]:
quotes = driver.find_elements(By.CLASS_NAME, 'quote')

datas = []

for first in quotes:
    text = first.find_element(By.CSS_SELECTOR, '.text').text
    author = first.find_element(By.CSS_SELECTOR, '.author').text
    link = first.find_elements(By.CSS_SELECTOR, 'span > a')[1].get_attribute('href')
    tags = first.find_elements(By.CSS_SELECTOR, '.tags > a')
    tag_text = ', '.join(tag.text for tag in tags)

    datas.append({
        'text': text,
        'author': author,
        'link': link,
        'tags': tag_text
    })

In [ ]:
df = pd.DataFrame(datas)

df

##### 다음 페이지 이동

In [ ]:
# 다음 페이지 정보 확인
next_button = driver.find_element(By.CSS_SELECTOR, 'li.next a')
next_button.click()

##### 여러 페이지 수집하기

In [ ]:
rows = []

driver.get('https://quotes.toscrape.com/')

for page in range(1, 6):
    wait.until(
        EC.presence_of_element_located((By.CLASS_NAME, 'footer'))
    )

    quotes = driver.find_elements(By.CSS_SELECTOR, '.quote')

    for qoute in quotes: # 한 페이지당 10개를 하나씩 반복
        text = qoute.find_element(By.CSS_SELECTOR, '.text').text
        author = qoute.find_element(By.CSS_SELECTOR, '.author').text
        link = qoute.find_elements(By.CSS_SELECTOR, 'span > a')[1].get_attribute('href')
        tags = qoute.find_elements(By.CSS_SELECTOR, '.tags > a')
        tag_text = ', '.join(tag.text for tag in tags)

        rows.append({
            'text': text,
            'author': author,
            'link': link,
            'tags': tag_text
        })

    next_button = driver.find_element(By.CSS_SELECTOR, 'li.next a')
    if not next_button:
        break
    next_button.click()

In [ ]:
len(rows)

In [ ]:
df_quotes = pd.DataFrame(rows)

df_quotes

##### 중복치 제거

In [ ]:
df_quotes = df_quotes.drop_duplicates(subset=['text', 'author'])

In [ ]:
df_quotes.info()

In [ ]:
# CSV 저장
df_quotes.to_csv('./data/quotes_to_scrap_50.csv', encoding='utf-8-sig')


In [ ]:
# 컬럼이름을 변경
df_quotes = df_quotes.rename(columns={'text': 'text', 
                          'author': 'author',
                          'link':'link', 
                          'tags':'tags'})

In [ ]:
df_quotes.head()